# Прогнозирование волатильности BTC-USDT

В ноутбуке собран полный воспроизводимый пайплайн проекта:
- загрузка часовых данных KuCoin;
- подготовка признаков;
- baseline-модель GARCH(1,1);
- DL-модель LSTM;
- walk-forward валидация без утечки данных;
- применение прогноза волатильности в vol-targeting стратегии;
- расчёт Sharpe, Max Drawdown и Monte Carlo сценариев.

## 1. Импорт библиотек и подготовка путей
Сначала подключим код проекта и посмотрим, где лежат артефакты полного прогона.

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

CONFIG_PATH = ROOT / 'config' / 'btc_kucoin_hourly.json'
SUMMARY_PATH = ROOT / 'reports' / 'summary.json'
PRED_PATH = ROOT / 'reports' / 'predictions.csv'
FORECAST_METRICS_PATH = ROOT / 'reports' / 'forecast_metrics.csv'
STRATEGY_METRICS_PATH = ROOT / 'reports' / 'strategy_metrics.csv'
FEATURE_PATH = ROOT / 'data' / 'processed' / 'feature_frame.csv'
FORECAST_PLOT = ROOT / 'reports' / 'forecast_comparison.png'
EQUITY_PLOT = ROOT / 'reports' / 'equity_curve.png'
MONTE_CARLO_PLOT = ROOT / 'reports' / 'monte_carlo_volatility.png'


## 2. Полный прогон проекта
Следующая ячейка запускает полный pipeline. Если артефакты уже построены, можно её не выполнять повторно.

In [ ]:
from ml_for_stock_market.pipeline import run_pipeline

# summary = run_pipeline(str(CONFIG_PATH))
# print(json.dumps(summary, indent=2, ensure_ascii=False))

## 3. Загрузка готовых результатов
Подгрузим summary, метрики и подготовленный датасет.

In [ ]:
summary = json.loads(SUMMARY_PATH.read_text(encoding='utf-8'))
predictions = pd.read_csv(PRED_PATH, parse_dates=['timestamp'])
forecast_metrics = pd.read_csv(FORECAST_METRICS_PATH)
strategy_metrics = pd.read_csv(STRATEGY_METRICS_PATH)
feature_frame = pd.read_csv(FEATURE_PATH, parse_dates=['timestamp'])

summary

## 4. Проверка данных
Посмотрим на объём выборки и на первые строки подготовленного признакового датасета.

In [ ]:
print('Rows in feature frame:', len(feature_frame))
print('Date range:', feature_frame['timestamp'].min(), '->', feature_frame['timestamp'].max())
display(feature_frame.head())

## 5. Качество прогноза волатильности
Сравним baseline GARCH и LSTM по основным ошибкам прогнозирования.

In [ ]:
display(forecast_metrics)

## 6. Метрики применения модели
Ниже показаны метрики для стратегии с управлением риском через прогноз волатильности.

In [ ]:
display(strategy_metrics)

## 7. Графики
Покажем график фактической и прогнозной волатильности, а также кривую капитала стратегии.

In [ ]:
display(Image(filename=str(FORECAST_PLOT)))
display(Image(filename=str(EQUITY_PLOT)))
display(Image(filename=str(MONTE_CARLO_PLOT)))

## 8. Краткие выводы
1. LSTM лучше baseline GARCH по RMSE, MAE и корреляции с реализованной волатильностью.
2. В прикладной части прогноз волатильности использовался для vol-targeting, а не для предсказания направления цены.
3. Monte Carlo даёт диапазон возможной будущей волатильности и помогает в сценарном риск-анализе.